# MuZero Implementation Comparison

This notebook benchmarks various MuZero implementations and configurations available in the codebase on the TicTacToe environment.

In [ ]:
import sys
import time
import torch
import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../../")
from agents.trainers.muzero_trainer import MuZeroTrainer
from configs.agents.muzero import MuZeroConfig
from configs.games.tictactoe import TicTacToeConfig
from agents.random import RandomAgent
from agents.tictactoe_expert import TicTacToeBestAgent
from modules.world_models.muzero_world_model import MuzeroWorldModel
from stats.stats import StatTracker

device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
base_params = {
    "num_simulations": 25,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "n_step": 10,
    "root_dirichlet_alpha": 0.25,
    "backbone": {
        "type": "resnet",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
    },
    "reward_head": {
        "neck": {"type": "resnet", "filters": [16], "kernel_sizes": [1], "strides": [1]},
        "output_strategy": {"type": "regression"},
    },
    "value_head": {
        "neck": {"type": "resnet", "filters": [16], "kernel_sizes": [1], "strides": [1]},
        "output_strategy": {"type": "regression"},
    },
    "policy_head": {
        "neck": {"type": "resnet", "filters": [16], "kernel_sizes": [1], "strides": [1]}
    },
    "known_bounds": [-1, 1],
    "minibatch_size": 8,
    "replay_buffer_size": 100000,
    "training_steps": 2000,
    "transfer_interval": 1,
    "num_workers": 4,
    "world_model_cls": MuzeroWorldModel,
    "multi_process": False
}

game_config = TicTacToeConfig()
env = game_config.make_env()

In [ ]:
variants = {}

# 1. Standard MuZero
variants["Standard"] = base_params.copy()
variants["Standard"]["model_name"] = "muzero_standard"

# 2. Batched Search
variants["Batched Search"] = base_params.copy()
variants["Batched Search"]["search_batch_size"] = 5
variants["Batched Search"]["use_virtual_mean"] = True
variants["Batched Search"]["model_name"] = "muzero_batched"

# 3. Value Prefix
variants["Value Prefix"] = base_params.copy()
variants["Value Prefix"]["value_prefix"] = True
variants["Value Prefix"]["model_name"] = "muzero_value_prefix"

# 4. EfficientZero
variants["EfficientZero"] = base_params.copy()
variants["EfficientZero"]["value_prefix"] = True
variants["EfficientZero"]["consistency_loss_factor"] = 2.0
variants["EfficientZero"]["model_name"] = "muzero_efficient_zero"

# 5. EfficientZero V2 (EfficientZero + Reanalyze)
variants["EfficientZero V2"] = base_params.copy()
variants["EfficientZero V2"]["value_prefix"] = True
variants["EfficientZero V2"]["consistency_loss_factor"] = 2.0
variants["EfficientZero V2"]["reanalyze_ratio"] = 0.8
variants["EfficientZero V2"]["model_name"] = "muzero_efficient_zero_v2"

# 6. MuZero Unplugged
variants["MuZero Unplugged"] = base_params.copy()
variants["MuZero Unplugged"]["reanalyze_ratio"] = 1.0
variants["MuZero Unplugged"]["injection_frac"] = 0.25
variants["MuZero Unplugged"]["model_name"] = "muzero_unplugged"

# 7. Sample MuZero
variants["Sample MuZero"] = base_params.copy()
variants["Sample MuZero"]["atom_size"] = 51
variants["Sample MuZero"]["support_range"] = 1
variants["Sample MuZero"]["model_name"] = "muzero_sampled"

# 8. Gumbel MuZero
variants["Gumbel MuZero"] = base_params.copy()
variants["Gumbel MuZero"]["gumbel"] = True
variants["Gumbel MuZero"]["gumbel_m"] = 16
variants["Gumbel MuZero"]["model_name"] = "muzero_gumbel"

# 9. Stochastic MuZero
variants["Stochastic MuZero"] = base_params.copy()
variants["Stochastic MuZero"]["stochastic"] = True
variants["Stochastic MuZero"]["vqvae_commitment_cost_factor"] = 0.25
variants["Stochastic MuZero"]["model_name"] = "muzero_stochastic"

In [ ]:
results = {}

for name, p in variants.items():
    print(f"\n--- Training variant: {name} ---")
    config = MuZeroConfig(config_dict=p, game_config=game_config)
    
    # Ensure explicit fields are set if needed (e.g. search_batch_size might be overridden by config init)
    if "search_batch_size" in p:
        config.search_batch_size = p["search_batch_size"]
    
    tracker = StatTracker(model_name=p["model_name"])
    trainer = MuZeroTrainer(
        config,
        env,
        device,
        stats=tracker,
        test_agents=[RandomAgent(), TicTacToeBestAgent()],
    )
    trainer.checkpoint_interval = 500
    trainer.test_interval = 1000
    trainer.test_trials = 100

    start_time = time.time()
    trainer.train()
    end_time = time.time()
    
    duration = end_time - start_time
    print(f"{name} Time: {duration:.2f}s")
    results[name] = {
        "duration": duration,
        "stats": tracker
    }

In [ ]:
plt.figure(figsize=(12, 8))
for name, data in results.items():
    stats = data["stats"].stats.get("test_score", {})
    if "avg" in stats and len(stats["avg"]) > 0:
        plt.plot(stats["avg"], label=name)
plt.title("MuZero Variants Comparison - Test Score")
plt.xlabel("Test Interval")
plt.ylabel("Avg Score")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 8))
names = list(results.keys())
durations = [data["duration"] for data in results.values()]
plt.bar(names, durations)
plt.title("Training Duration Comparison")
plt.ylabel("Time (s)")
plt.xticks(rotation=45)
plt.show()